# TD 9 : un RAG complet, de bout en bout, jusqu'à une interface Gradio

## Comment se passe ce TD

Vous travaillez **seul, après le cours**, à votre rythme. Il n'y a personne à qui demander, et le notebook est construit pour cela.

**Chaque exercice se termine par une cellule de vérification** qui vous dit si vous avez juste, et ce qui ne va pas sinon. Les indices sont repliés sous chaque exercice : ouvrez le premier avant de chercher ailleurs, le second est presque la réponse.

**Vous pouvez vous arrêter entre deux étapes.** En reprenant, relancez toutes les cellules depuis le début : les étapes s'appuient sur ce que les précédentes ont calculé.

## Ce que vous saurez faire

1. construire un corpus de recherche à partir de données réelles et mal rangées
2. mesurer ce qu'un encodeur lit vraiment, et décider s'il faut découper
3. comparer trois façons de chercher sur trois sortes de questions
4. décider du refus avant la génération, avec un seuil mesuré
5. mesurer ce qu'une phrase de la consigne change
6. livrer une interface Gradio et la tester par son API
7. écrire la fiche d'un système, limites comprises

## Prérequis

- avoir fait les TD 7 et 8
- avoir téléchargé les deux modèles du jour 3 avec `telecharger_modeles.py`
- avoir installé les dépendances du `requirements.txt` de cette remise, qui ajoute `gradio`

**Durée :** 240 minutes

## Le contexte

Le TD 8 a montré qu'un RAG s'assemble en une après-midi, et qu'il se mesure. Celui-ci construit le système complet, sur **les 7 484 avis de la HAS** et non plus sur 1 500, jusqu'à une interface qu'un utilisateur peut ouvrir dans son navigateur.

À chaque étape, une décision d'ingénierie, et la mesure qui la justifie. Plusieurs résultats vont contredire l'intuition : c'est pour cela qu'on mesure.

## Le déroulé

| quand | quoi |
|---|---|
| 0:00 | étape 1 : le cahier des charges |
| 0:15 | étape 2 : le corpus réel |
| 0:45 | étape 3 : ce que l'encodeur lit vraiment |
| 1:00 | étape 4 : trois façons de chercher |
| 1:40 | étape 5 : le refus, avant la génération |
| 2:10 | étape 6 : le système routé |
| 2:30 | étape 7 : générer, et ce que la consigne change |
| 3:05 | étape 8 : l'interface Gradio |
| 3:45 | étape 9 : la fiche du système |
| 4:00 | fin : un système, et ses limites écrites |

> ### Avant de commencer
>
> Le système que vous allez construire est une **démonstration d'ingénierie**. Ce n'est ni un dispositif médical ni un conseil thérapeutique, et il ne doit jamais être présenté comme tel, y compris dans votre portfolio.
>
> Le corpus a été choisi pour cela : un avis de service médical rendu porte sur le **remboursement** d'un médicament, pas sur sa posologie.
>
> L'interface de l'étape 8 tourne **sur votre machine seulement**. Ne la publiez pas avec un lien public : `share=False` reste à `False`.

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

In [ ]:
# Fourni, rien a faire : les reglages du TD.
import os, re, time, difflib
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'   # Gradio n'envoie aucune statistique

ENCODEUR = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
GENERATEUR = 'Qwen/Qwen2.5-1.5B-Instruct'
NIVEAUX = ['Important', 'Insuffisant', 'Modéré', 'Faible']

SORTIES = racine / 'sorties'      # le journal et la fiche du systeme iront ici
SORTIES.mkdir(exist_ok=True)
print('sorties :', SORTIES)

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# dedupliquer sur deux colonnes
t = t.drop_duplicates(subset=['a', 'b'])

# une matrice TF-IDF sur des documents, puis les scores d'une question
vec = TfidfVectorizer()
X = vec.fit_transform(textes)
s = (X @ vec.transform([question]).T).toarray().ravel()

# des n-grammes de caracteres, robustes aux fautes
TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)

# encoder, normalise, pour que le produit scalaire soit un cosinus
V = encodeur.encode(textes, normalize_embeddings=True, batch_size=64)

# le rang de chaque document, du meilleur au moins bon
ordre = np.argsort(-s)
rang = np.empty(len(s)); rang[ordre] = np.arange(1, len(s) + 1)

# le nom connu le plus proche, et sa proximite d'edition
proche = difflib.get_close_matches(nom, noms, n=1, cutoff=0.0)[0]
ratio = difflib.SequenceMatcher(None, nom, proche).ratio()

# une interface Gradio minimale
import gradio as gr
with gr.Blocks() as demo:
    q = gr.Textbox(label='Question')
    r = gr.Textbox(label='Reponse')
    gr.Button('Chercher').click(fonction, q, r, api_name='repondre')
demo.launch(prevent_thread_lock=True, share=False)
```

---

## Étape 1 : Le cahier des charges

**15 minutes.** Avant la première ligne de code : pour qui, pour quoi, et comment saurez-vous que cela marche.

Un système de recherche se juge sur ce qu'il devait faire. Si vous ne l'écrivez pas maintenant, vous le jugerez sur ce qu'il fait, et il aura toujours raison.

Remplissez les cinq rubriques. Il n'y a pas de réponse unique : la vérification contrôle seulement qu'aucune n'est vide et que vos métriques sont mesurables.

In [ ]:
# A FAIRE : le cahier des charges, en quelques mots par rubrique.
cahier = {
    'utilisateur': ...,   # qui pose les questions ?
    'questions': ...,     # lesquelles ? deux exemples
    'si_inconnu': ...,    # que doit faire le systeme quand il ne sait pas ?
    'metriques': ...,     # comment saurez-vous qu'il marche ? des choses que l'on compte
    'interdit': ...,      # ce qu'il ne doit jamais faire
}

**Indice 1.** Pensez à un pharmacien d'hôpital qui prépare une commission et veut le SMR d'un médicament, ou à un étudiant qui cherche quels médicaments ont été évalués dans une maladie.

**Indice 2.** Pour les métriques : la part de questions dont le bon avis arrive en tête, la part de noms inventés refusés, le temps de réponse.

In [ ]:
vides = [k for k, v in cahier.items() if v is ... or not v]
assert not vides, f'rubriques vides : {vides}'
texte = str(cahier['metriques']).lower()
assert any(m in texte for m in ('hit', 'refus', 'precision', 'part', 'temps', '%')), (
    'les metriques doivent se compter : une part, un temps, un hit@k...')
print('cahier des charges ecrit. L etape 9 le confrontera aux mesures.')

---

## Étape 2 : Le corpus réel

**30 minutes.** Les données ne sont pas rangées comme on le voudrait. On le mesure avant de décider quoi indexer.

In [ ]:
# Fourni, rien a faire : le fichier brut.
avis = pd.read_parquet(DONNEES / 'avis_has.parquet')
print(f"{len(avis):,} lignes, {avis['code_has'].nunique():,} avis distincts")
avis[['code_has', 'date_avis', 'smr', 'texte']].head(3)

### 2.1 Qu'est-ce qu'un document ?

Une ligne du fichier est un couple (médicament, avis) : un même avis couvre souvent plusieurs présentations, et son texte est alors répété. Mais un avis peut aussi porter **plusieurs SMR**, un par indication, avec un texte par SMR. Le document à indexer est donc le couple (avis, texte).

In [ ]:
# A FAIRE : un document par couple (code_has, texte), aux quatre niveaux de SMR.
docs = avis[avis['smr'].isin(NIVEAUX)].copy()
docs['texte'] = docs['texte'].str.strip()
docs = ...        # sans doublon sur code_has ET texte, puis reset_index(drop=True)
docs['date'] = pd.to_datetime(docs['date_avis'], format='%Y%m%d')

plusieurs = docs.groupby('code_has')['smr'].nunique()
print(f'{len(docs):,} documents, issus de {docs["code_has"].nunique():,} avis')
print(f'{(plusieurs > 1).sum():,} avis portent plusieurs SMR')

**Indice 1.** Le premier patron du mémo, sur les colonnes `code_has` et `texte`.

**Indice 2.** `docs = docs.drop_duplicates(subset=['code_has', 'texte']).reset_index(drop=True)`

In [ ]:
assert len(docs) == 8853, (
    f'attendu 8 853 documents, obtenu {len(docs):,}. Avez-vous filtre les quatre '
    'niveaux, puis deduplique sur code_has ET texte ?')
assert (plusieurs > 1).sum() == 1224, f'attendu 1 224, obtenu {(plusieurs > 1).sum()}'
print('correct : 8 853 documents, et 1 224 avis portent plusieurs SMR.')
print()
print('Indexer un avis entier melangerait des indications au SMR different. Le')
print('document est ce que la reponse citera : il doit porter UN seul SMR.')

### 2.2 Le nom du médicament

Pour filtrer par médicament à l'étape 6, il faut le nom dans une colonne. Le motif du TD 8 le prenait après « rendu par ». Celui-ci accepte aussi « les spécialités », les noms composés, et s'arrête au premier chiffre : une question dit « STRUCTUM », pas « STRUCTUM 500 ».

In [ ]:
# A FAIRE : mesurer la couverture des deux motifs, et regarder ce qui manque.
MOTIF_TD8 = re.compile(r"rendu par ([A-ZÉÈÀÇ][A-Z0-9ÉÈÀÇ\-\.' ]{2,40})")
MOTIF = re.compile(r"rendu par (?:les |la )?(?:spécialités? )?"
                   r"([A-ZÉÈÀÇ][A-ZÉÈÀÇ\-']{2,}(?: (?:[A-ZÉÈÀÇ][A-ZÉÈÀÇ\-']+))*)")

docs['medicament'] = (docs['texte'].str.extract(MOTIF, expand=False)
                                   .str.strip().str.rstrip(" -'"))
couverture_td8 = ...     # part des documents ou MOTIF_TD8 trouve quelque chose
couverture = ...         # part des documents qui ont un medicament
print(f'motif du TD 8 : {couverture_td8:.1%}   motif du TD 9 : {couverture:.1%}')
print(f"{docs['medicament'].nunique():,} medicaments distincts")
print()
for t in docs.loc[docs['medicament'].isna(), 'texte'].head(4):
    print('-', t[:110])

**Indice 1.** `str.extract(...)` rend `NaN` quand le motif ne trouve rien. La moyenne de `.notna()` est une part.

**Indice 2.** `couverture_td8 = docs['texte'].str.extract(MOTIF_TD8, expand=False).notna().mean()` et `couverture = docs['medicament'].notna().mean()`

In [ ]:
assert abs(couverture_td8 - 0.682) < 0.002, f'motif du TD 8 : attendu 68,2 %'
assert abs(couverture - 0.737) < 0.002, f'motif du TD 9 : attendu 73,7 %'
assert docs['medicament'].nunique() == 3592
print('correct : 68,2 % avec le motif du TD 8, 73,7 % avec celui-ci.')
print()
print('Plus d un quart des documents n ont toujours pas de nom : « Les donnees de')
print('l etude... », « A completer apres A/R »... Ils restent cherchables par leur')
print('texte, mais le filtre par medicament de l etape 6 ne les verra jamais.')
print('C est une limite du systeme, et elle ira dans sa fiche.')

---

## Étape 3 : Ce que l'encodeur lit vraiment

**15 minutes.** Un encodeur a une longueur maximale. Au-delà, il tronque, sans prévenir.

In [ ]:
# Fourni, rien a faire : l'encodeur des TD 7 et 8.
from sentence_transformers import SentenceTransformer
encodeur = SentenceTransformer(ENCODEUR)
tok_enc = encodeur.tokenizer
print('longueur maximale lue :', encodeur.max_seq_length, 'tokens')

In [ ]:
# A FAIRE : la longueur de chaque document en tokens, et ce qui depasse 128.
longueurs = np.array([...])       # len(tok_enc(t)['input_ids']) pour chaque texte
trop = longueurs > 128
perdus = (longueurs[trop] - 128).sum() / longueurs.sum()
print(f'mediane {np.median(longueurs):.0f} tokens, max {longueurs.max()}')
print(f'{trop.sum():,} documents depassent 128 tokens ({trop.mean():.1%})')
print(f"part du texte que l'encodeur ne lit pas : {perdus:.1%}")

**Indice 1.** Une compréhension de liste sur `docs['texte']`.

**Indice 2.** `longueurs = np.array([len(tok_enc(t)['input_ids']) for t in docs['texte']])`

In [ ]:
assert trop.sum() == 713, f'attendu 713 documents, obtenu {trop.sum()}'
assert abs(perdus - 0.097) < 0.002
print("correct : 713 documents, 8,1 %, et 9,7 % du texte que l'encodeur ignore.")
print()
print('Un document de 300 tokens est encode sur ses 128 premiers. Si la reponse est')
print('a la fin, la recherche dense ne la verra pas, et rien ne le signale.')

### 3.1 Découper ce qui dépasse

On découpe seulement les documents trop longs, en fenêtres de 100 tokens qui se recouvrent de 20, pour qu'une phrase coupée en deux se retrouve entière dans l'une des deux fenêtres. Chaque morceau garde le numéro de son document.

In [ ]:
# A FAIRE : completer la boucle de decoupage.
def morceaux_de(textes, taille=100, recouvrement=20):
    out = []
    for i, t in enumerate(textes):
        ids = tok_enc(t, add_special_tokens=False)['input_ids']
        if len(ids) <= 126:            # tient dans l'encodeur avec ses deux jetons speciaux
            out.append((t, i))
            continue
        pas = taille - recouvrement
        for debut in range(0, len(ids), pas):
            out.append(...)            # le texte de la fenetre, et le numero du document
            if debut + taille >= len(ids):
                break
    return out

morceaux = morceaux_de(list(docs['texte']))
print(f'{len(morceaux):,} morceaux pour {len(docs):,} documents')

**Indice 1.** Une fenêtre est `ids[debut:debut + taille]`, qu'il faut redécoder en texte avec le tokenizer.

**Indice 2.** `out.append((tok_enc.decode(ids[debut:debut + taille]), i))`

In [ ]:
assert len(morceaux) == 10078, f'attendu 10 078 morceaux, obtenu {len(morceaux):,}'
assert {d for _, d in morceaux} == set(range(len(docs))), 'un document a ete perdu'
print('correct : 10 078 morceaux, et chaque document est represente.')
print("L'etape 4 dira si ce decoupage change quelque chose. Ne le supposez pas.")

---

## Étape 4 : Trois façons de chercher

**40 minutes.** Le lexical, le dense, et leur fusion, mesurés sur trois sortes de questions.

### 4.1 Le jeu d'évaluation, fourni

Quatre sortes de questions, parce qu'un système se juge sur ce qu'on lui demandera vraiment :

- **40 noms exacts** : « Quel est le service médical rendu de STERIDOSE ? » ;
- **les 40 mêmes, mal tapés** : deux lettres voisines échangées ;
- **10 noms inventés**, qui n'existent pas ;
- **18 paraphrases** en langage courant, sans le mot médical : « un excès de sucre dans le sang ». Un document est jugé pertinent s'il contient le terme médical, ici « diabète ».

In [ ]:
# Fourni, rien a faire : le jeu d'evaluation.
def faute(nom):
    """Deux lettres voisines echangees au milieu du nom."""
    i = len(nom) // 2
    return nom[:i - 1] + nom[i] + nom[i - 1] + nom[i + 1:]

INVENTES = ['ZYPHORAXINE', 'MELCORVIDOL', 'TRANEXIBOR', 'CALDREXIMOL', 'VORANTELIDE',
            'PREXILURANE', 'DOMIRAXEN', 'FELTOZANIB', 'QUINALOPRIDE', 'SOVERANTIL']
PARAPHRASES = [
    ('hypertension', 'une tension trop forte dans les artères'),
    ('diabète', 'un excès de sucre dans le sang'),
    ('polyarthrite', 'une inflammation chronique de plusieurs articulations'),
    ('asthme', 'des crises de gêne respiratoire avec sifflements'),
    ('psoriasis', 'des plaques rouges qui desquament sur la peau'),
    ('sclérose en plaques', 'une maladie qui attaque la gaine des nerfs'),
    ('ostéoporose', 'des os devenus fragiles'),
    ('dépress', 'une tristesse pathologique qui dure'),
    ('insuffisance cardiaque', 'un cœur qui ne pompe plus assez'),
    ('hypercholestérolémie', 'trop de cholestérol dans le sang'),
    ('VIH', 'le virus du sida'),
    ('épilepsie', 'des crises convulsives à répétition'),
    ('insomnie', 'des difficultés à dormir'),
    ('schizophrénie', 'une psychose chronique avec hallucinations'),
    ('mucoviscidose', "une maladie génétique qui épaissit le mucus des poumons"),
    ('hémophilie', 'un trouble héréditaire de la coagulation'),
    ('migraine', 'des maux de tête intenses et récurrents'),
    ('parkinson', "des tremblements et une raideur d'origine neurologique"),
]

rng = np.random.default_rng(42)
par_nom = docs['medicament'].value_counts()
uniques = sorted(n for n in par_nom[par_nom == 1].index if len(n) >= 6 and ' ' not in n)
noms_eval = list(rng.choice(uniques, size=40, replace=False))
assert not set(INVENTES) & set(docs['medicament'].dropna())

exacts = [{'type': 'nom', 'nom': n, 'question': f'Quel est le service médical rendu de {n} ?',
           'doc': int(docs.index[docs['medicament'] == n][0])} for n in noms_eval]
fautes = [{**e, 'type': 'faute',
           'question': f"Quel est le service médical rendu de {faute(e['nom'])} ?"}
          for e in exacts]
inventes = [{'type': 'invente', 'nom': n, 'doc': None,
             'question': f'Quel est le service médical rendu de {n} ?'} for n in INVENTES]
para = []
for terme, dit in PARAPHRASES:
    motif = 'VIH|immunodéficience humaine' if terme == 'VIH' else re.escape(terme)
    para.append({'terme': terme,
                 'question': f'Quels médicaments ont été évalués pour {dit} ?',
                 'pertinents': set(np.flatnonzero(
                     docs['texte'].str.contains(motif, case=False, regex=True)))})
print(f'{len(exacts)} noms exacts, {len(fautes)} fautes, {len(inventes)} inventes, '
      f'{len(para)} paraphrases')
print('exemple de faute :', exacts[0]['nom'], '->', faute(exacts[0]['nom']))

### 4.2 Les deux TF-IDF

Le TF-IDF sur les **mots**, celui du TD 7, et un TF-IDF sur les **n-grammes de caractères**, qui découpe « STERIDOSE » en « ste », « ter », « eri »... et reconnaît donc encore un nom dont deux lettres ont bougé.

In [ ]:
# A FAIRE : les deux vectoriseurs, ajustes sur les DOCUMENTS.
from sklearn.feature_extraction.text import TfidfVectorizer
textes = list(docs['texte'])

vec_mots = TfidfVectorizer()
X_mots = vec_mots.fit_transform(textes)
vec_car = ...
X_car = ...

def scores_mots(question):
    return (X_mots @ vec_mots.transform([question]).T).toarray().ravel()

def scores_car(question):
    return ...

print(X_mots.shape, X_car.shape)

**Indice 1.** Les patrons 2 et 3 du mémo. `scores_car` est `scores_mots` avec `vec_car` et `X_car`.

**Indice 2.** `vec_car = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)`, `X_car = vec_car.fit_transform(textes)`, et `return (X_car @ vec_car.transform([question]).T).toarray().ravel()`

### 4.3 Le dense, sur les documents puis sur les morceaux

Deux encodages : les 8 853 documents entiers, puis les 10 078 morceaux. Avec les morceaux, un document vaut le score de son **meilleur** morceau. Comptez une à deux minutes.

In [ ]:
# A FAIRE : encoder les documents, puis les morceaux.
debut = time.perf_counter()
V_docs = ...
V_morceaux = ...
proprio = np.array([d for _, d in morceaux])     # le document de chaque morceau
print(f'encodage : {time.perf_counter() - debut:.0f}s')

def scores_dense(question, V=None, proprio=None):
    V = V_docs if V is None else V
    q = encodeur.encode([question], normalize_embeddings=True, show_progress_bar=False)[0]
    s = V @ q
    if proprio is None:
        return s
    out = np.full(len(docs), -1.0)
    np.maximum.at(out, proprio, s)                # un document vaut son meilleur morceau
    return out

**Indice 1.** Le patron 4 du mémo, deux fois : sur `textes`, puis sur les textes des morceaux. Ajoutez `show_progress_bar=False`.

**Indice 2.** `V_docs = encodeur.encode(textes, normalize_embeddings=True, batch_size=64, show_progress_bar=False)` et la même chose sur `[m for m, _ in morceaux]`

### 4.4 La fusion, et la mesure

La **reciprocal rank fusion** additionne, pour chaque document, `1 / (60 + rang)` dans chaque classement. Elle ne demande aucun réglage, et c'est la fusion la plus courante. On fusionne le TF-IDF sur caractères et le dense.

In [ ]:
# A FAIRE : la fusion par les rangs.
def scores_hybride(question):
    total = np.zeros(len(docs))
    for s in (scores_car(question), scores_dense(question)):
        ordre = np.argsort(-s)
        rang = ...                 # le rang de chaque document, 1 pour le meilleur
        total += 1.0 / (60 + rang)
    return total

SCORES = {'tfidf_mots': scores_mots, 'tfidf_car': scores_car,
          'dense': scores_dense, 'hybride': scores_hybride}

def top(scores, k=5):
    return np.argsort(-scores)[:k]

def evaluer(fonctions):
    res = {}
    for nom, f in fonctions.items():
        ligne = {}
        for jeu_nom, jeu in (('nom', exacts), ('faute', fautes)):
            rangs = np.array([int(np.flatnonzero(np.argsort(-f(e['question'])) == e['doc'])[0]) + 1
                              for e in jeu])
            ligne[jeu_nom + '_hit1'] = int((rangs <= 1).sum())
            ligne[jeu_nom + '_hit5'] = int((rangs <= 5).sum())
            ligne[jeu_nom + '_mrr'] = round(float(np.mean(1 / rangs)), 3)
        ligne['para_p5'] = round(float(np.mean(
            [len(set(top(f(e['question']))) & e['pertinents']) / 5 for e in para])), 3)
        res[nom] = ligne
    return pd.DataFrame(res).T

resultats = evaluer(SCORES)
resultats

**Indice 1.** Le patron 5 du mémo : on crée un tableau vide, puis on écrit les rangs 1, 2, 3... aux positions données par `ordre`.

**Indice 2.** `rang = np.empty(len(s)); rang[ordre] = np.arange(1, len(s) + 1)`

In [ ]:
r = resultats
assert r.loc['tfidf_car', 'nom_hit1'] == 36, r.loc['tfidf_car', 'nom_hit1']
assert r.loc['dense', 'nom_hit1'] == 6, r.loc['dense', 'nom_hit1']
assert r.loc['tfidf_mots', 'faute_hit5'] == 0
assert r.loc['tfidf_car', 'faute_hit1'] == 19
assert abs(r.loc['dense', 'para_p5'] - 0.478) < 0.006
assert abs(r.loc['tfidf_mots', 'para_p5'] - 0.033) < 0.006
assert r.loc['hybride', 'nom_hit1'] == 13
print('correct. Trois resultats, et deux contredisent l intuition :')
print()
print('1. Sur un NOM, le lexical ecrase le dense : 36/40 contre 6/40. Un nom de')
print('   medicament est un mot rare et exact, et c est ce que TF-IDF sait faire.')
print('2. Sur une PARAPHRASE, c est l inverse : P@5 de 0,478 pour le dense contre')
print('   0,033 pour TF-IDF sur les mots, qui ne voit aucun mot commun.')
print('3. La fusion ne prend pas le meilleur des deux : 13/40 sur les noms, bien')
print('   moins que les 36/40 du lexical seul. Le dense tire les noms vers le bas.')

In [ ]:
# Fourni, rien a faire : et le decoupage, change-t-il quelque chose ?
avec_morceaux = evaluer({'dense, morceaux': lambda q: scores_dense(q, V_morceaux, proprio)})
pd.concat([resultats.loc[['dense']], avec_morceaux])

In [ ]:
p5 = avec_morceaux.loc['dense, morceaux', 'para_p5']
assert abs(p5 - 0.500) < 0.006, p5
print(f'paraphrases P@5 : 0,478 sans decoupage, {p5:.3f} avec.')
print()
print('Le decoupage aide un peu, et seulement sur les paraphrases : 92 % des documents')
print('tiennent deja dans l encodeur, et les noms sont dans les premiers tokens. On le')
print('garde, parce qu il ne coute presque rien, mais on sait maintenant ce qu il vaut.')

---

## Étape 5 : Le refus, avant la génération

**30 minutes.** Décider qu'il n'y a pas de réponse, avec une mesure qui sépare vraiment.

Le TD 8 l'a montré : la recherche ne refuse jamais, elle classe. Le refus doit donc venir d'une décision. On essaie trois mesures, sur les 40 noms exacts, les 40 fautes et les 10 noms inventés, et on garde celle qui sépare.

In [ ]:
# Fourni, rien a faire : extraire le nom d'une question, et la proximite par
# n-grammes de caracteres sur la liste des noms connus.
def extraire_nom(question):
    """La plus longue suite de mots en majuscules, de quatre lettres au moins."""
    trouves = re.findall(r"[A-ZÉÈÀÇ][A-ZÉÈÀÇ\-']{3,}(?: [A-ZÉÈÀÇ][A-ZÉÈÀÇ\-']+)*", question)
    return max(trouves, key=len) if trouves else None

NOMS = sorted(set(docs['medicament'].dropna()))
vec_noms = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_noms = vec_noms.fit_transform(NOMS)

def proximite_car(nom):
    s = (X_noms @ vec_noms.transform([nom]).T).toarray().ravel()
    return NOMS[int(np.argmax(s))], float(s.max())

print(extraire_nom('Quel est le service médical rendu de STEIRDOSE ?'))
print(proximite_car('STEIRDOSE'))

In [ ]:
# A FAIRE : la proximite d'edition, puis les trois mesures sur les 90 questions.
def proximite_edition(nom):
    proche = ...
    return proche, difflib.SequenceMatcher(None, nom, proche).ratio()

lignes = []
for e in exacts + fautes + inventes:
    nom = extraire_nom(e['question'])
    connu, edition = proximite_edition(nom)
    lignes.append({'type': e['type'],
                   'dense': float(scores_dense(e['question']).max()),
                   'ngrammes': proximite_car(nom)[1],
                   'edition': edition,
                   'corrige': connu == e['nom']})
mesures = pd.DataFrame(lignes)
mesures.groupby('type')[['dense', 'ngrammes', 'edition']].agg(['min', 'median', 'max']).round(3)

**Indice 1.** Le patron 6 du mémo : le nom connu le plus proche dans `NOMS`.

**Indice 2.** `proche = difflib.get_close_matches(nom, NOMS, n=1, cutoff=0.0)[0]`

In [ ]:
g = mesures.groupby('type')
assert g['dense'].median()['invente'] > 0.74, 'le dense devrait trouver les inventes proches'
assert g['ngrammes'].max()['invente'] > g['ngrammes'].min()['faute'], (
    'les n-grammes devraient melanger fautes et inventes')
assert g['edition'].max()['invente'] < g['edition'].min()['faute'], (
    'la proximite d edition devrait separer les deux')
print("correct. Similarite dense : les inventes ont la meme mediane que les vrais noms,")
print("0,749 contre 0,752. Elle ne peut rien decider.")
print("N-grammes : les fautes descendent a 0,300, les inventes montent a 0,467. Melanges.")
print("Edition : les fautes restent au-dessus de 0,833, les inventes sous 0,700.")
print()
print("Une faute de frappe est une EDITION. La bonne mesure est celle qui correspond a")
print("l'erreur que l'on attend.")

In [ ]:
# A FAIRE : le seuil, au milieu de l'ecart entre fautes et inventes.
seuil = ...
corriges = int(mesures.loc[mesures['type'] == 'faute', 'corrige'].sum())
print(f'seuil : {seuil}')
print(f'fautes corrigees vers le bon nom : {corriges}/40')

**Indice 1.** Le plus haut score d'un nom inventé, le plus bas d'une faute, et leur moyenne, arrondie à deux décimales.

**Indice 2.** `seuil = round((g['edition'].max()['invente'] + g['edition'].min()['faute']) / 2, 2)`

In [ ]:
assert seuil == 0.77, f'attendu 0,77, obtenu {seuil}'
assert corriges == 40, f'attendu 40/40, obtenu {corriges}'
print('correct : seuil 0,77, et les 40 fautes sont corrigees vers le bon nom.')
print()
print('Attention a ce que ce seuil vaut : il est choisi sur ces 90 questions, et rien')
print('ne garantit qu il separe aussi bien les suivantes. Il faudrait le valider sur')
print('un autre jeu. Cela ira dans la fiche du systeme.')

---

## Étape 6 : Le système routé

**20 minutes.** Au lieu de fusionner, on aiguille chaque question vers ce qui lui convient.

Trois routes, décidées par la question elle-même :

1. **un nom, mais aucun nom connu assez proche** : on refuse, sans rien générer ;
2. **un nom connu, exact ou corrigé** : on **filtre par la métadonnée** `medicament`, l'avis le plus récent d'abord ;
3. **pas de nom** : la recherche dense, sur les morceaux.

In [ ]:
# A FAIRE : la fonction de recherche du systeme.
def chercher(question, k=3):
    nom = extraire_nom(question)
    if nom is None:
        s = scores_dense(question, V_morceaux, proprio)
        return {'route': 'sens', 'nom': None, 'top': top(s, k)}
    connu, edition = proximite_edition(nom)
    if ...:                                          # aucun nom connu assez proche
        return {'route': 'refus', 'nom': nom, 'top': np.array([], dtype=int)}
    siens = docs[docs['medicament'] == connu]
    recents = ...                                    # les k plus recents d'abord
    return {'route': 'nom', 'nom': connu, 'top': np.array(recents.index)}

debut = time.perf_counter()
decisions = {t: [chercher(e['question']) for e in jeu]
             for t, jeu in (('nom', exacts), ('faute', fautes), ('invente', inventes))}
latence = (time.perf_counter() - debut) / 90
for t, jeu in (('nom', exacts), ('faute', fautes)):
    bons = sum(len(d['top']) > 0 and d['top'][0] == e['doc']
               for d, e in zip(decisions[t], jeu))
    print(f'{t:<8} bon avis en tete : {bons}/40')
refuses = sum(d['route'] == 'refus' for d in decisions['invente'])
print(f'inventes refuses avant generation : {refuses}/10')
p5_systeme = np.mean([len(set(chercher(e['question'], 5)['top']) & e['pertinents']) / 5
                      for e in para])
print(f'paraphrases P@5 : {p5_systeme:.3f}')
print(f'latence de la recherche : {latence * 1000:.0f} ms')

**Indice 1.** Le refus compare `edition` au `seuil` de l'étape 5. Le filtre trie par la colonne `date`, de la plus récente à la plus ancienne.

**Indice 2.** `if edition < seuil:` et `recents = siens.sort_values('date', ascending=False).head(k)`

In [ ]:
bons_nom = sum(d['top'][0] == e['doc'] for d, e in zip(decisions['nom'], exacts))
bons_faute = sum(d['top'][0] == e['doc'] for d, e in zip(decisions['faute'], fautes))
assert bons_nom == 40 and bons_faute == 40, (bons_nom, bons_faute)
assert refuses == 10, refuses
assert abs(p5_systeme - 0.500) < 0.006
print('correct : 40/40 sur les noms exacts, 40/40 sur les fautes, 10/10 refus,')
print('et P@5 de 0,500 sur les paraphrases.')
print()
print('Soyez honnete sur le 40/40 : les questions ont ete fabriquees avec les memes')
print('noms que le filtre utilise. Il prouve que la tuyauterie marche, pas que le')
print('systeme trouve tout. Les documents sans nom, plus d un quart, lui echappent.')

---

## Étape 7 : Générer, et ce que la consigne change

**35 minutes.** Quelques mots de consigne déplacent les refus et les réponses justes. On le mesure, sur les deux sortes de questions.

In [ ]:
# Fourni, rien a faire : le generateur du TD 8, et trois consignes.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(GENERATEUR)
generateur = AutoModelForCausalLM.from_pretrained(GENERATEUR, dtype=torch.float32).eval()

# sobre : le refus d'abord, puis une phrase qui donne le niveau
CONSIGNE_SOBRE = (
    "Tu reponds UNIQUEMENT a partir des extraits fournis. "
    "Si les extraits ne contiennent pas la reponse, tu reponds exactement : "
    "\"Information absente des documents fournis.\" "
    "Sinon, reponds en une phrase qui donne le niveau de service medical rendu.")
# citation : un modele de phrase, et le numero de l'extrait entre crochets
CONSIGNE_CITATION = (
    "Tu reponds UNIQUEMENT a partir des extraits numerotes fournis. "
    "Reponds en une seule phrase de la forme : Le service medical rendu de <medicament> "
    "est <niveau> [<numero de l'extrait>]. "
    "Si les extraits ne permettent pas de repondre, reponds exactement : "
    "\"Information absente des documents fournis.\"")
# forme : le refus d'abord, puis le modele de phrase, sans citation
CONSIGNE = (
    "Tu reponds UNIQUEMENT a partir des extraits fournis. "
    "Si les extraits ne contiennent pas la reponse, tu reponds exactement : "
    "\"Information absente des documents fournis.\" "
    "Sinon, reponds en une seule phrase de la forme : "
    "Le service medical rendu de <medicament> est <niveau>.")
CONSIGNES = {'sobre': CONSIGNE_SOBRE, 'citation': CONSIGNE_CITATION, 'forme': CONSIGNE}
REFUS = 'Information absente des documents fournis.'

def generer(question, extraits, consigne=CONSIGNE):
    contexte = '\n\n'.join(f'[{i}] {t}' for i, t in enumerate(extraits, start=1))
    messages = [{'role': 'system', 'content': consigne},
                {'role': 'user', 'content': f'Extraits :\n{contexte}\n\nQuestion : {question}'}]
    entree = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    jetons = tok(entree, return_tensors='pt')
    with torch.no_grad():
        sortie = generateur.generate(**jetons, max_new_tokens=90, do_sample=False,
                                     pad_token_id=tok.eos_token_id)
    return tok.decode(sortie[0][jetons['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def refuse(reponse):
    bas = reponse.lower()
    return 'absente des documents' in bas or 'aucune information' in bas

def citations(reponse):
    return [int(x) for x in re.findall(r'\[(\d)\]', reponse)]

print('generateur pret')

### 7.1 Sans garde-fou, combien de refus ?

On retire le garde-fou, et l'on donne au générateur les trois avis les plus proches de chaque nom inventé, comme le ferait une recherche lexicale. Les trois consignes ne diffèrent que de quelques mots. Comptez deux à trois minutes.

In [ ]:
# A FAIRE : le nombre de refus sur les 10 noms inventes, pour chaque consigne.
refus_par_consigne = {}
for nom_c, consigne in CONSIGNES.items():
    n = 0
    for e in inventes:
        extraits = [docs['texte'].iloc[i] for i in top(scores_car(e['question']), 3)]
        n += ...
    refus_par_consigne[nom_c] = n
refus_par_consigne

**Indice 1.** `refuse(...)` d'une réponse générée avec cette consigne.

**Indice 2.** `n += refuse(generer(e['question'], extraits, consigne))`

In [ ]:
r = refus_par_consigne
assert abs(r['sobre'] - 7) <= 1, r
assert r['citation'] <= 1, r
assert abs(r['forme'] - 5) <= 1, r
for k, v in r.items():
    print(f'{k:<9}: {v} refus sur 10')
print()
print('Demander une citation au format [n] a fait disparaitre les refus : le modele a')
print('appris a toujours citer quelque chose. Et le modele de phrase de la consigne')
print('forme en coute aussi. Quelques mots, et le systeme invente sur des noms qui')
print('n existent pas.')

### 7.2 Et sur 30 questions réelles ?

Vingt noms exacts et dix fautes, par le système routé. Une consigne qui refuse bien peut répondre mal : il faut mesurer les deux. Comptez deux à trois minutes par consigne.

In [ ]:
# A FAIRE : evaluer une consigne sur les 30 questions reelles.
choix = exacts[:20] + fautes[:10]

def evaluer_consigne(consigne):
    justes = citees = 0
    durees = []
    for e in choix:
        d = chercher(e['question'])
        question = e['question'].replace(extraire_nom(e['question']), d['nom'])
        debut = time.perf_counter()
        rep = ...                    # generer, avec les textes de d['top']
        durees.append(time.perf_counter() - debut)
        justes += docs['smr'].iloc[e['doc']].lower()[:5] in rep.lower()
        citees += bool(citations(rep))
    return {'justes': justes, 'citees': citees,
            'mediane_s': round(float(np.median(durees)), 1)}

bilan = {'forme': evaluer_consigne(CONSIGNE)}
bilan

**Indice 1.** Les extraits sont `[docs['texte'].iloc[i] for i in d['top']]`.

**Indice 2.** `rep = generer(question, [docs['texte'].iloc[i] for i in d['top']], consigne)`

In [ ]:
# Fourni, rien a faire : la consigne sobre.
bilan['sobre'] = evaluer_consigne(CONSIGNE_SOBRE)
bilan['sobre']

In [ ]:
# Fourni, rien a faire : la consigne qui demande une citation.
bilan['citation'] = evaluer_consigne(CONSIGNE_CITATION)
tableau = pd.DataFrame(bilan).T
tableau['refus sur 10 inventes'] = pd.Series(refus_par_consigne)
tableau

In [ ]:
assert bilan['forme']['justes'] >= 30 - 1, bilan
assert abs(bilan['sobre']['justes'] - 24) <= 1, bilan
assert bilan['citation']['citees'] <= 2 + 1, bilan
print(tableau[['justes', 'citees', 'refus sur 10 inventes']])
print()
print('Aucune consigne ne fait tout. La sobre refuse le mieux et repond le moins juste.')
print('La forme repond juste et refuse moins. La citation ne refuse plus, et ne cite')
print('presque jamais, alors qu on le lui demande.')
print()
print('D ou le partage des roles du systeme : le garde-fou de l etape 5, deterministe,')
print('arrete les noms inconnus, 10 sur 10. La consigne est choisie pour repondre juste :')
print('la forme. Et les sources sont attachees par le code, pas demandees au modele.')

---

## Étape 8 : L'interface Gradio

**40 minutes.** Une interface qu'un utilisateur peut ouvrir, un journal de ce qu'il demande, et un test par l'API.

La fonction derrière l'interface fait tout le système : elle cherche, refuse ou génère, attache les sources, et **écrit chaque question dans un journal**. Le journal des refus est la liste de ce que vos documents ne couvrent pas.

In [ ]:
# A FAIRE : la fonction de l'interface.
import csv
from datetime import datetime

JOURNAL = SORTIES / 'journal_td9.csv'

def source_lisible(i):
    nom = docs['medicament'].iloc[i]
    nom = nom if isinstance(nom, str) else f"avis {docs['code_has'].iloc[i]}"
    return f"{nom}, avis du {docs['date'].iloc[i]:%d/%m/%Y}, SMR {docs['smr'].iloc[i].lower()}"

def repondre_interface(question):
    debut = time.perf_counter()
    d = chercher(question)
    if d['route'] == 'refus':
        reponse = f"{REFUS} Aucun médicament ne s'appelle {d['nom']} dans le corpus."
        sources = ''
    else:
        q = question if d['nom'] is None else question.replace(extraire_nom(question), d['nom'])
        reponse = ...
        sources = ...            # une ligne par source : [1] NOM, avis du ..., SMR ...
    with open(JOURNAL, 'a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([datetime.now().isoformat(timespec='seconds'), question,
                                d['route'], d['nom'] or '',
                                d['route'] == 'refus' or refuse(reponse),
                                round(time.perf_counter() - debut, 1)])
    return reponse, sources

print(repondre_interface(inventes[0]['question'])[0])

**Indice 1.** La réponse est `generer(q, extraits)` avec les textes de `d['top']`. Les sources joignent `source_lisible(i)` par des retours à la ligne.

**Indice 2.** `reponse = generer(q, [docs['texte'].iloc[i] for i in d['top']])` et `sources = '\n'.join(f'[{k}] {source_lisible(i)}' for k, i in enumerate(d['top'], start=1))`

### 8.1 L'interface

Une zone de question, des exemples cliquables, la réponse, les sources, et l'avertissement. `share=False` : l'interface n'est visible que depuis votre machine, à l'adresse que Gradio affiche.

In [ ]:
# A FAIRE : completer l'interface, puis la lancer.
import gradio as gr

AVERTISSEMENT = ("Démonstration d'ingénierie sur les avis de la HAS. Ni dispositif médical, "
                 "ni conseil thérapeutique : un avis de SMR porte sur le remboursement.")

with gr.Blocks(title='Avis de la HAS') as demo:
    gr.Markdown('## Rechercher dans les avis de la HAS')
    gr.Markdown(AVERTISSEMENT)
    question = gr.Textbox(label='Question', lines=2)
    gr.Examples([exacts[0]['question'], fautes[1]['question'], inventes[0]['question'],
                 para[1]['question']], inputs=question)
    bouton = gr.Button('Chercher', variant='primary')
    reponse = gr.Textbox(label='Réponse', lines=3)
    sources = gr.Textbox(label='Sources', lines=3)
    ...                      # relier le bouton a repondre_interface, api_name='repondre'

demo.launch(prevent_thread_lock=True, server_name='127.0.0.1', share=False, quiet=True)
print('interface :', demo.local_url)

**Indice 1.** Le dernier patron du mémo : `bouton.click(fonction, entrées, sorties, api_name=...)`. Il y a une entrée et deux sorties.

**Indice 2.** `bouton.click(repondre_interface, question, [reponse, sources], api_name='repondre')`

### 8.2 Le test par l'API

Une interface se teste comme un utilisateur s'en sert : on lui envoie des questions par son API, et on vérifie ce qui revient. Puis ouvrez l'adresse affichée dans votre navigateur, et essayez vos propres questions.

In [ ]:
from gradio_client import Client

client = Client(demo.local_url, verbose=False)
essais = {
    'nom exact': exacts[0]['question'],
    'nom mal tape': fautes[1]['question'],
    'nom invente': inventes[0]['question'],
    'paraphrase': para[1]['question'],
    'sigle': 'Quels médicaments ont été évalués dans la BPCO ?',
}
retours = {k: client.predict(q, api_name='/repondre') for k, q in essais.items()}
for k, (rep, src) in retours.items():
    print(f'{k:<13} -> {rep[:90]}')
    if src:
        print(f"{'':<16}{src.splitlines()[0]}")

assert REFUS in retours['nom invente'][0] and not retours['nom invente'][1]
assert exacts[1]['nom'] in retours['nom mal tape'][1], 'la faute devrait etre corrigee'
assert retours['paraphrase'][1].count('[') == 3, 'trois sources attendues'
assert REFUS in retours['sigle'][0], 'le sigle devrait etre pris pour un nom'
journal = pd.read_csv(JOURNAL, header=None,
                      names=['quand', 'question', 'route', 'nom', 'refus', 'secondes'])
print()
print(f'journal : {len(journal)} questions, dont {int(journal["refus"].sum())} refus')
print()
print('Le dernier essai est un echec du systeme, et il faut le voir : « BPCO » est en')
print('majuscules, donc pris pour un nom de medicament, et refuse. Le routage par')
print('majuscules a un angle mort. Il ira dans la fiche du systeme.')

---

## Étape 9 : La fiche du système

**15 minutes.** Ce que le système fait, ce qu'il vaut, mesuré, et ce qu'il ne fait pas.

Une fiche de système dit trois choses : à quoi il sert, ce qu'il vaut **mesuré**, et ce qu'il ne fait pas. La deuxième partie s'écrit toute seule à partir de vos variables. La troisième, c'est vous.

In [ ]:
# A FAIRE : au moins quatre limites, chacune appuyee sur ce que vous avez vu.
limites = [
    ...,
]

fiche = f"""# Fiche du système : recherche dans les avis de la HAS

## À quoi il sert
{cahier['utilisateur']}. Questions : {cahier['questions']}.

## Ce qu'il vaut, mesuré
- corpus : {len(docs):,} documents, dont {couverture:.1%} avec un nom de médicament
- noms exacts : {bons_nom}/40 bon avis en tête ; noms mal tapés : {bons_faute}/40
- noms inventés refusés avant génération : {refuses}/10, seuil {seuil}
- paraphrases : précision des 5 premiers {p5_systeme:.3f}
- réponses justes : {bilan['forme']['justes']}/30 ; génération médiane {bilan['forme']['mediane_s']} s

## Ce qu'il ne fait pas
""" + "\n".join(f"- {l}" for l in limites) + f"""

## Usage
{AVERTISSEMENT}
"""
(SORTIES / 'fiche_systeme.md').write_text(fiche, encoding='utf-8')
print(fiche)

**Indice 1.** Relisez les messages des vérifications : les documents sans nom, le seuil choisi sur 90 questions, le 40/40 fabriqué, les sigles en majuscules, la citation que le modèle ne fait pas, un corpus figé à une date.

**Indice 2.** Une limite utile dit ce qui casse et dans quel cas : « un sigle en majuscules, comme BPCO, est pris pour un nom de médicament et refusé ».

In [ ]:
assert all(isinstance(l, str) and len(l) > 20 for l in limites), (
    'chaque limite est une phrase qui dit ce qui casse, et dans quel cas')
assert len(limites) >= 4, f'au moins quatre limites, vous en avez {len(limites)}'
assert (SORTIES / 'fiche_systeme.md').exists()
print(f'fiche ecrite : {SORTIES / "fiche_systeme.md"}')
print()
print('Relisez votre cahier des charges de l etape 1 a cote de la fiche. Ce que vous')
print('vouliez mesurer l a-t-il ete ? Ce qui manque est le travail suivant.')
print()
print("Quand vous avez fini : demo.close() ferme l'interface.")

## Ce que cette étape a établi

- Un document est ce que la réponse citera : ici le couple (avis, texte), parce qu'un avis peut porter plusieurs SMR.
- L'encodeur lit 128 tokens et tronque le reste. Le découpage se décide sur la mesure, et ici il n'apporte presque rien.
- Le lexical gagne sur les noms, le dense sur les paraphrases, et une fusion naïve fait moins bien que le meilleur des deux. Router vaut mieux que fusionner.
- Le refus se décide avant la génération, avec la mesure qui correspond à l'erreur attendue : une faute de frappe est une édition.
- Une phrase de consigne peut supprimer tous les refus. Ce que le code peut garantir, le refus d'un nom inconnu, la source affichée, ne se confie pas au modèle.
- Une interface se teste par son API, et un système se livre avec sa fiche, limites comprises.

## Les erreurs classiques

- **Évaluer la recherche sur un seul type de question.** Les noms et les paraphrases donnent des classements opposés.
- **Fusionner deux méthodes parce que c'est la pratique courante.** Mesurez la fusion contre chacune des deux. Ici elle perd.
- **Régler la consigne sur les questions qui marchent.** Re-mesurez le refus à chaque modification de la consigne.
- **Publier l'interface avec share=True.** Une capture suffit pour un portfolio, et un déploiement se protège.

## Pour aller plus loin, si vous avez fini en avance

- Corriger l'angle mort des sigles : ne router vers le filtre que si le nom ressemble à un nom connu, ajouter cinq questions à sigles au jeu d'évaluation, et re-mesurer tout le tableau.
- Essayer une fusion pondérée, avec un poids plus fort pour le lexical. Existe-t-il un poids qui bat le routage sur les deux types de questions ?
- Remplacer le motif du nom par une liste de médicaments officielle, et mesurer combien de documents sans nom en récupèrent un.
- Faire de ce TD le volet « lire et interroger » de votre projet portfolio : la fiche du système en est le livrable.